In [1]:
import papermill as pm
import duckdb, subprocess, os, time
from joblib import Parallel, delayed
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")
start_time = time.time()
print(time.ctime(time.time()))

Tue Feb 17 13:40:25 2026


In [2]:
# setup parameters for sensitivity analysils
'''
static run

migration waterfowl curves - flatten the curves a few different ways. Set curves static as 1 to 0.5 in increments of 0.1
Do this for all species.  With and without geese.

Goose reduction percentage - 0, 25, 50, 75, 1

habitat availability - Make it all available the entire time[100].  Available early [100,50,25] in the season vs 
mid [33,100,33], vs late [25,50,100].

removewater = True/False

remove moist soil

Reduce/Increase kcalperduck by 10/20%
'''
listfips = {'default':[], 'goosereductionpct':[0,25,75,100], 'remove_water':[], 'remove_moistsoil':[],'low_moistsoil':[],'high_moistsoil':[],'kcalperduck':[236, 265.5, 324.5, 354], 'energy_early':[100,50,25], 
            'energy_mid':[33,100,33], 'energy_late':[25,50,100], 'migration_curve':[0.5, 0.6, 0.7, 0.8, 0.9, 1]}

In [3]:
# Define function for running
def f(x):
    print(x)
    os.makedirs('./output/', exist_ok=True)
    os.makedirs('./output/plots/', exist_ok=True)
    os.makedirs('./output/notebooks/', exist_ok=True)
    try:
        if x == 'default':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x), parameters=dict(),shell=True))
        elif x == 'goosereductionpct':
            for pct in listfips[x]:
                if pct == 0:
                    subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), geese=False)), shell=True)
                else:
                    subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), goosereductionpct=pct)), shell=True)
        elif x == 'remove_water':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, removewater=True)), shell=True)
        elif x == 'remove_moistsoil':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, removemoistsoil=True)), shell=True)
        elif x == 'low_moistsoil':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, lowmoistsoil=True)), shell=True)
        elif x == 'high_moistsoil':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, highmoistsoil=True)), shell=True)            
        elif x == 'kcalperduck':
            for pct in listfips[x]:
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), kcalperduck=pct)), shell=True)
        elif x == 'energy_early':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=listfips[x])), shell=True)
        elif x == 'energy_mid':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=listfips[x])), shell=True)   
        elif x == 'energy_late':
            subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=listfips[x])), shell=True)
        elif x == 'migration_curve':
            for pct in listfips[x]:
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/notebooks/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), changecurvepct=pct)), shell=True)     
        return (x, 'run complete')
    except Exception as e:
        return (x,'fail', e)

In [4]:
# Run analysis
completed = Parallel(n_jobs=4, verbose=10)(delayed(f)(key) for key in listfips)
print('########## Complete ##########')
print(f'Time elapsed: {time.time() - start_time} seconds')

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


remove_water
default
goosereductionpct
remove_moistsoil


Executing: 100%|██████████| 61/61 [01:19<00:00,  1.30s/cell]
metadata: 1: cells: not found
Executing:   0%|          | 0/61 [00:00<?, ?cell/s]

low_moistsoil


Executing: 100%|██████████| 61/61 [01:24<00:00,  1.39s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:27<00:00,  1.43s/cell]
metadata: 1: cells: not found
Executing:   0%|          | 0/61 [00:00<?, ?cell/s]

...

Executing: 100%|██████████| 61/61 [01:25<00:00,  1.39s/cell]
metadata: 1: cells: not found
Executing:   0%|          | 0/61 [00:00<?, ?cell/s]

migration_curve


Executing: 100%|██████████| 61/61 [01:21<00:00,  1.34s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:24<00:00,  1.39s/cell]
metadata: 1: cells: not found
[Parallel(n_jobs=4)]: Done   8 out of  11 | elapsed:  5.6min remaining:  2.1min
Executing: 100%|██████████| 61/61 [01:23<00:00,  1.36s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:22<00:00,  1.35s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:18<00:00,  1.28s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:11<00:00,  1.17s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:11<00:00,  1.18s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:10<00:00,  1.15s/cell]
metadata: 1: cells: not found
Executing: 100%|██████████| 61/61 [01:12<00:00,  1.19s/cell]


########## Complete ##########
Time elapsed: 707.4175729751587 seconds


metadata: 1: cells: not found
[Parallel(n_jobs=4)]: Done  11 out of  11 | elapsed: 11.8min finished


In [5]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe'
folder_path = 'output'

# Collect data from all matching CSV files
data_frames = []

for filename in os.listdir(folder_path):
    if filename.endswith('outputresults.csv'):
        file_path = os.path.join(folder_path, filename)
        df = pd.read_csv(file_path)
        if 'statebcr' in df.columns and 'leftoverLo' in df.columns:
            df['source'] = filename[:-18]
            data_frames.append(df)

# Combine all data
combined_df = pd.concat(data_frames)

# Sort by statebcr for consistent ordering
combined_df['statebcr'] = combined_df['statebcr'].astype(str)  # Ensure categorical
combined_df.sort_values(by='statebcr', inplace=True)

# Create base figure
fig = px.bar(
    combined_df,
    x='statebcr',
    y='leftoverLo',
    color='source',
    barmode='group',
    title='Leftover KCals by State BCR for sensitivity analysis',
    labels={'statebcr': 'State BCR', 'leftoverLo': 'Leftover KCals', 'source': 'Source File'},
    hover_data=['source', 'leftoverLo'],
    height=1000
)
fig.update_layout(xaxis_tickangle=90)
# Add vertical lines between each statebcr category
unique_statebcrs = combined_df['statebcr'].unique()
for i in range(1, len(unique_statebcrs)):
    center = i - 0.5
    fig.add_vline(x=i - 0.5, line_width=1.5, line_dash="solid", line_color="black")
    fig.add_vline(x=center - 0.05, line_width=1, line_dash="solid", line_color="white")
    fig.add_vline(x=center + 0.05, line_width=1, line_dash="solid", line_color="white")

fig.show()
# Save and show
fig.write_html(os.path.join('output','Sensitivity_leftoverenergy.html'))
